# 11.2 — Four Formats Side by Side

**Chapter 11, §3.2 "CSV, JSON, Avro, and Parquet" and Table 11.2 (and Exercise 3).**

Table 11.2 makes four structural claims about each format: where the schema lives, whether the
file is row- or column-oriented, whether it is text or binary, and where it can be split. This
notebook writes **one** ten-row table four ways and then checks each claim against the bytes on
disk.

The chapter's §3.1 warning is the thing to keep in view: row orientation and textual encoding are
*independent* axes. Avro is the case that proves it — row-oriented and binary at once.

Chapter 3 owns Parquet itself. Nothing here re-explains row groups or pruning; Parquet appears
only as one of four points in the design space.

In [1]:
import glob, json, os, shutil, tempfile
from pyspark.sql import SparkSession

SCRATCH = os.path.join(tempfile.gettempdir(), "cs777", "ch11-formats")
shutil.rmtree(SCRATCH, ignore_errors=True)
os.makedirs(SCRATCH)

# The Avro data source is not part of the PySpark distribution. Requesting the package here
# means this notebook needs no special launch command; the first run fetches it from Maven
# Central and later runs use the Ivy cache.
spark = (
    SparkSession.builder.appName("CS777-11.2")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-avro_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", 4)
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark.version

:: loading settings :: url = jar:file:/Users/dt/.venvs/cs777-spark/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/dt/.ivy2.5.2/cache
The jars for the packages stored in: /Users/dt/.ivy2.5.2/jars
org.apache.spark#spark-avro_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-50d33466-6872-4432-894e-835b5694d98c;1.0
	confs: [default]
	found org.apache.spark#spark-avro_2.13;4.2.0 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 in central
	found org.tukaani#xz;1.12 in central
:: resolution report :: resolve 77ms :: artifacts dl 2ms
	:: modules in use:
	org.apache.spark#spark-avro_2.13;4.2.0 from central in [default]
	org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 from central in [default]
	org.tukaani#xz;1.12 from central in [default]
	---------------------------------------------------------------------
	|            

'4.2.0'

In [2]:
users = spark.createDataFrame(
    [
        (1, "ada",    "red",   1979),
        (2, "alan",   "blue",  1936),
        (3, "grace",  "green", 1959),
        (4, "edsger", "blue",  1968),
        (5, "barbara","red",   1969),
        (6, "donald", "green", 1968),
        (7, "john",   "blue",  1945),
        (8, "claude", "red",   1948),
        (9, "tim",    "green", 1989),
        (10, "linus", "blue",  1991),
    ],
    "id int, name string, color string, year int",
)
users.show()

+---+-------+-----+----+
| id|   name|color|year|
+---+-------+-----+----+
|  1|    ada|  red|1979|
|  2|   alan| blue|1936|
|  3|  grace|green|1959|
|  4| edsger| blue|1968|
|  5|barbara|  red|1969|
|  6| donald|green|1968|
|  7|   john| blue|1945|
|  8| claude|  red|1948|
|  9|    tim|green|1989|
| 10|  linus| blue|1991|
+---+-------+-----+----+



## 1. One table, four files

Each format is written with one output file so that "the file" below is unambiguous. The
compression is turned off for Parquet and Avro, because the question here is where the bytes are,
not how few of them there are.

In [3]:
paths = {fmt: os.path.join(SCRATCH, fmt) for fmt in ("csv", "json", "avro", "parquet")}
one = users.coalesce(1)

one.write.option("header", True).csv(paths["csv"])
one.write.json(paths["json"])
one.write.format("avro").option("compression", "uncompressed").save(paths["avro"])
one.write.option("compression", "none").parquet(paths["parquet"])


def data_file(fmt, ext):
    # The one part file Spark wrote, ignoring _SUCCESS and the .crc sidecars.
    return glob.glob(os.path.join(paths[fmt], f"part-*{ext}"))[0]


files = {
    "csv": data_file("csv", ".csv"),
    "json": data_file("json", ".json"),
    "avro": data_file("avro", ".avro"),
    "parquet": data_file("parquet", ".parquet"),
}
for fmt, path in files.items():
    print(f"{fmt:8} {os.path.getsize(path):>6} bytes")

csv         199 bytes
json        510 bytes
avro        490 bytes
parquet    1344 bytes


## 2. Where the schema lives

Four different answers, all visible in the first bytes of the file.

In [4]:
def head(path, n=160):
    with open(path, "rb") as f:
        return f.read(n)


print("CSV  :", head(files["csv"], 60))
print()
print("JSON :", head(files["json"], 120))

CSV  : b'id,name,color,year\n1,ada,red,1979\n2,alan,blue,1936\n3,grace,g'

JSON : b'{"id":1,"name":"ada","color":"red","year":1979}\n{"id":2,"name":"alan","color":"blue","year":1936}\n{"id":3,"name":"grace"'


**CSV** carries one header line of *names* and no types at all: `id` could be an integer, a
string, or a date. Table 11.2's "None (external)" means that the schema has to arrive from
somewhere else — a reader option, a registered table, or a guess produced by sampling.

**JSON** repeats every field name in every record. That is a schema of a sort, and it is why the
format tolerates records that differ from one another, but it is paid for once per row.

In [5]:
print("AVRO :", head(files["avro"], 230))

AVRO : b'Obj\x01\x06\x16avro.schema\xa2\x03{"type":"record","name":"topLevelRecord","fields":[{"name":"id","type":["int","null"]},{"name":"name","type":["string","null"]},{"name":"color","type":["string","null"]},{"name":"year","type":["int","null"]}]}0o'


**Avro** opens with the four magic bytes `Obj\x01` and then a metadata map whose `avro.schema`
entry is the full schema **as JSON, once, in the header**. Everything after the header is binary:
no field names, no separators, just values in schema order. That is the combination the chapter
asks the reader to notice — the schema is textual and self-describing, the data is not.

In [6]:
print("PARQUET first 4 bytes:", head(files["parquet"], 4))
with open(files["parquet"], "rb") as f:
    f.seek(-4, os.SEEK_END)
    print("PARQUET last  4 bytes:", f.read())

PARQUET first 4 bytes: b'PAR1'
PARQUET last  4 bytes: b'PAR1'


**Parquet** is `PAR1` at both ends and nothing readable in between. The schema is in the footer,
and the four trailing bytes are the magic that lets a reader find it: seek to the end, read the
footer length, jump back, and the structure of the whole file is known before any data page is
touched.

In [7]:
import pyarrow.parquet as pq

pf = pq.ParquetFile(files["parquet"])
print(pf.schema)
print("row groups:", pf.num_row_groups, " rows:", pf.metadata.num_rows)

required group field_id=-1 spark_schema {
  optional int32 field_id=-1 id;
  optional binary field_id=-1 name (String);
  optional binary field_id=-1 color (String);
  optional int32 field_id=-1 year;
}

row groups: 1  rows: 10


## 3. Row-contiguous against column-contiguous

The claim of §3.1 is about *where values sit relative to each other*. Avro stores the four fields
of one record together; Parquet stores all ten values of one column together.

Parquet says so in its own metadata: each column chunk is a byte range, and the ranges do not
interleave.

In [8]:
rg = pf.metadata.row_group(0)
print(f"{'column':<8} {'first byte':>11} {'bytes':>7}")
for i in range(rg.num_columns):
    col = rg.column(i)
    print(f"{col.path_in_schema:<8} {col.data_page_offset:>11} {col.total_compressed_size:>7}")

column    first byte   bytes
id                 4      69
name              73     120
color            236      78
year             271      69


Four disjoint runs, one per column, laid end to end. A query that needs `color` reads one of
those runs and never touches the other three — the column pruning chapter 3 developed, here as
an arithmetic fact about offsets.

The row-oriented formats report no such structure, so the contrast has to be read off the bytes
themselves. Two distances say it: how far `ada` is from `red` — two fields of the **same record**
— and how far `ada` is from `alan` — two values of the **same column**.

In [9]:
def gap(raw, a, b):
    return raw.find(b) - raw.find(a)


print(f"{'':8} {'ada -> red':>12} {'ada -> alan':>13}")
print(f"{'':8} {'(one record)':>12} {'(one column)':>13}")
for fmt in ("csv", "json", "avro", "parquet"):
    raw = open(files[fmt], "rb").read()
    print(f"{fmt:8} {gap(raw, b'ada', b'red'):>12} {gap(raw, b'ada', b'alan'):>13}")

           ada -> red   ada -> alan
         (one record)  (one column)
csv                 4            15
json               14            48
avro                5            15
parquet           108             7


The three row-oriented files put the fields of one record within a few bytes of each other and
push the next value of the same column a whole record away. Parquet inverts both numbers: the two
names are adjacent, and the color belonging to the same record is on the far side of an entire
column chunk.

One caution about reading raw bytes this way. `red` occurs a second time in the Parquet file, in
the footer's min/max statistics for the `color` column, and a value repeated many times in a real
table would be stored once in a dictionary rather than once per row. Byte-hunting shows the
layout; the column-chunk offsets printed above are the instrument that actually measures it.

## 4. Where each format can be split

Splittability is what lets one file be read by several tasks at once, and each format announces
its boundaries differently.

In [10]:
raw_avro = open(files["avro"], "rb").read()

# An Avro file ends with the sync marker that terminates its last block, so the final 16
# bytes are the marker itself. Counting its occurrences counts the block boundaries.
sync = raw_avro[-16:]
print("sync marker:", sync.hex())
print("occurrences in the file:", raw_avro.count(sync))

sync marker: 82049a250e2d36bd945acc16253c41ba
occurrences in the file: 2


Two: one at the end of the header, one at the end of the single block this small file contains.
A reader handed a byte offset in the middle of a large Avro file scans forward for the next
occurrence of these sixteen bytes and starts there — which is exactly the mechanism §3.1
describes, and the reason the marker is random rather than a fixed constant.

In [11]:
print("parquet row groups        :", pf.num_row_groups, "(the split unit)")
print("json  newline-separated   :", open(files['json'], 'rb').read().count(b'\n'), "records")
print("csv   newline-separated   :", open(files['csv'], 'rb').read().count(b'\n') - 1,
      "records after the header")

parquet row groups        : 1 (the split unit)
json  newline-separated   : 10 records
csv   newline-separated   : 10 records after the header


Text formats split on the line, which is why the header row of a CSV is a nuisance: it is data to
every task but the first. Neither splits at all once wrapped in a non-splittable codec such as
gzip, which is the parenthesis in Table 11.2's CSV row.

## 5. Schema evolution

Table 11.2 claims formal schema resolution for Avro and schema merging for Parquet. Both are
checkable.

In [12]:
# A reader schema with a field the writer never wrote. Avro resolves the difference using
# the default declared in the reader's schema.
reader_schema = {
    "type": "record",
    "name": "topLevelRecord",
    "fields": [
        {"name": "id", "type": ["int", "null"]},
        {"name": "name", "type": ["string", "null"]},
        {"name": "color", "type": ["string", "null"]},
        {"name": "year", "type": ["int", "null"]},
        {"name": "country", "type": ["string", "null"], "default": None},
    ],
}
evolved = spark.read.format("avro").option("avroSchema", json.dumps(reader_schema)).load(paths["avro"])
evolved.show(3)

+---+-----+-----+----+-------+
| id| name|color|year|country|
+---+-----+-----+----+-------+
|  1|  ada|  red|1979|   NULL|
|  2| alan| blue|1936|   NULL|
|  3|grace|green|1959|   NULL|
+---+-----+-----+----+-------+
only showing top 3 rows


In [13]:
# Parquet instead merges the schemas of the files it finds, when asked to.
extra = spark.createDataFrame([(11, "jean", "red", 1930, "FR")],
                              "id int, name string, color string, year int, country string")
extra.coalesce(1).write.mode("append").parquet(paths["parquet"])

merged = spark.read.option("mergeSchema", "true").parquet(paths["parquet"])
print("country present after merge:", "country" in merged.columns)
merged.select("id", "name", "country").orderBy("id").show(11)

country present after merge: True


+---+-------+-------+
| id|   name|country|
+---+-------+-------+
|  1|    ada|   NULL|
|  2|   alan|   NULL|
|  3|  grace|   NULL|
|  4| edsger|   NULL|
|  5|barbara|   NULL|
|  6| donald|   NULL|
|  7|   john|   NULL|
|  8| claude|   NULL|
|  9|    tim|   NULL|
| 10|  linus|   NULL|
| 11|   jean|     FR|
+---+-------+-------+



The ten rows written without the column read back with a null in it, and the one row that has it
keeps it. Neither file was rewritten; the reader reconciled two footers.

CSV has no mechanism of either kind. A reader is given names by a header line and then applies
them **by position**, so a file with a fifth column read under a four-column schema simply loses
its fifth field, silently.

In [14]:
five_col = os.path.join(SCRATCH, "csv5")
extra.coalesce(1).write.option("header", True).csv(five_col)
print(open(glob.glob(os.path.join(five_col, "part-*.csv"))[0]).read())

(spark.read.option("header", True)
      .schema("id int, name string, color string, year int")
      .csv(five_col).show())

id,name,color,year,country
11,jean,red,1930,FR



+---+----+-----+----+
| id|name|color|year|
+---+----+-----+----+
| 11|jean|  red|1930|
+---+----+-----+----+



`FR` is gone and nothing reported it. That is what "None" in Table 11.2's schema-evolution column
means in practice, and it is why §3.2 says analytical pipelines convert text into a typed binary
representation immediately after ingestion.

## Conclusion

Table 11.2, verified line by line:

| | schema found at | layout | encoding | splits at |
|---|---|---|---|---|
| CSV | nowhere in the file — one header line of names, no types | row | text | newline |
| JSON | repeated in every record | row | text | newline |
| Avro | the header, as JSON, once | row | **binary** | a random 16-byte sync marker |
| Parquet | the footer, after the data | **column** | binary | row group |

The two axes really are independent: Avro is row-oriented *and* binary, which is why it suits an
event stream whose consumers each want whole records, while Parquet's disjoint column chunks are
what make an analytical scan read one field out of four.

In [15]:
spark.stop()